Grid search for hyperparameters using Optuna

In [12]:
import numpy as np
import pandas as pd
import optuna
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# =========================================================
# OPTUNA SEARCH FOR XGBOOST-CHOLESKY COVARIANCE MODEL
# Rank by average QLIKE, keep Frobenius too
# Validation period: 2019-2020
# Compare against realized covariance proxy
# Refit every 21 forecast origins
# =========================================================

# ---------------------------------------------------------
# SETTINGS
# ---------------------------------------------------------
TARGET_H = 21

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
PROXY_PATH = f"../proxy/realized_cov_h{TARGET_H}.csv"

ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

VALIDATION_START = "2019-01-01"
if TARGET_H == 10:
    VALIDATION_END   = "2020-12-17"
elif TARGET_H == 21:
    VALIDATION_END   = "2020-12-02"
elif TARGET_H == 63:
    VALIDATION_END   = "2020-10-02"

LAGS = (10, 21, 63)
REFIT_EVERY = 21

RIDGE_EPS = 1e-8
RANDOM_STATE = 42
N_JOBS = -1

# Number of Optuna trials
N_TRIALS = 40

# Save paths
SAVE_TRIALS_PATH = f"../results/xgb_optuna_trials_h{TARGET_H}.csv"
SAVE_BEST_PATH = f"../results/xgb_optuna_best_h{TARGET_H}.csv"

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
chol = (
    pd.read_csv(CHOL_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .reset_index(drop=True)
)

proxy = (
    pd.read_csv(PROXY_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .set_index("Date")
)

validation_dates = chol.loc[
    (chol["Date"] >= pd.Timestamp(VALIDATION_START)) &
    (chol["Date"] <= pd.Timestamp(VALIDATION_END)),
    "Date"
]

print("Loaded Cholesky data:", CHOL_PATH)
print("Loaded proxy data:", PROXY_PATH)
print("Validation:", validation_dates.min(), "->", validation_dates.max())
print("Refit every:", REFIT_EVERY)
print("Number of Optuna trials:", N_TRIALS)
print()

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs

def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]

def vec_to_matrix(vec, n_assets):
    M = np.asarray(vec, dtype=float).reshape(n_assets, n_assets)
    M = 0.5 * (M + M.T)
    return M

def frobenius_distance(S_true, H_pred):
    return float(np.linalg.norm(S_true - H_pred, ord="fro"))

def make_spd(M, eps=RIDGE_EPS):
    M = 0.5 * (np.asarray(M, dtype=float) + np.asarray(M, dtype=float).T)
    eigvals = np.linalg.eigvalsh(M)
    min_eig = eigvals.min()
    if min_eig <= eps:
        M = M + np.eye(M.shape[0]) * (eps - min_eig + eps)
    return M

def qlike_loss(S_true, H_pred, eps=RIDGE_EPS):
    """
    Matrix QLIKE:
        tr(S_true @ H_pred^{-1}) - logdet(S_true @ H_pred^{-1}) - n

    Computed in a numerically stable way as:
        tr(H_pred^{-1} S_true) - logdet(S_true) + logdet(H_pred) - n
    """
    S_true = make_spd(S_true, eps=eps)
    H_pred = make_spd(H_pred, eps=eps)

    n = S_true.shape[0]

    sign_s, logdet_s = np.linalg.slogdet(S_true)
    sign_h, logdet_h = np.linalg.slogdet(H_pred)

    if sign_s <= 0 or sign_h <= 0:
        return np.nan

    try:
        trace_term = np.trace(np.linalg.solve(H_pred, S_true))
    except np.linalg.LinAlgError:
        return np.nan

    return float(trace_term - logdet_s + logdet_h - n)

lower_pairs = build_lower_tri_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

proxy_cols = [f"cov_{a}__{b}" for a, b in full_pairs]

n_assets = len(ASSET_ORDER)
asset_to_idx = {a: i for i, a in enumerate(ASSET_ORDER)}

missing_proxy_cols = [c for c in proxy_cols if c not in proxy.columns]
if missing_proxy_cols:
    raise ValueError(f"Missing proxy columns, e.g. {missing_proxy_cols[:5]}")

# ---------------------------------------------------------
# MODEL EVALUATION FUNCTION
# ---------------------------------------------------------
def evaluate_xgb_spec(
    window,
    n_estimators,
    max_depth,
    learning_rate,
    subsample,
    colsample_bytree,
    min_child_weight,
    reg_lambda
):
    models = None
    scalers_x = None
    scalers_y = None
    last_refit_loc = None

    frob_list = []
    qlike_list = []
    n_forecasts = 0
    n_refits = 0

    for p in validation_dates:

        p_loc = chol.index[chol["Date"] == p][0]

        should_refit = (
            models is None
            or last_refit_loc is None
            or (p_loc - last_refit_loc) >= REFIT_EVERY
        )

        if should_refit:

            train_end = p_loc - TARGET_H
            train_start = train_end - window + 1

            if train_start < 0:
                continue

            train = chol.iloc[train_start:train_end + 1]

            models = {}
            scalers_x = {}
            scalers_y = {}

            for a, b in lower_pairs:

                target = f"target_chol_{a}__{b}"
                lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

                missing_cols = [c for c in lag_cols + [target] if c not in train.columns]
                if missing_cols:
                    raise ValueError(f"Missing columns for {(a, b)}: {missing_cols}")

                X = train[lag_cols].values
                y = train[target].values.reshape(-1, 1)

                scaler_x = StandardScaler()
                scaler_y = StandardScaler()

                Xs = scaler_x.fit_transform(X)
                ys = scaler_y.fit_transform(y).ravel()

                model = XGBRegressor(
                    objective="reg:squarederror",
                    n_estimators=n_estimators,
                    max_depth=max_depth,
                    learning_rate=learning_rate,
                    subsample=subsample,
                    colsample_bytree=colsample_bytree,
                    min_child_weight=min_child_weight,
                    reg_lambda=reg_lambda,
                    random_state=RANDOM_STATE,
                    n_jobs=N_JOBS
                )

                model.fit(Xs, ys)

                models[(a, b)] = model
                scalers_x[(a, b)] = scaler_x
                scalers_y[(a, b)] = scaler_y

            last_refit_loc = p_loc
            n_refits += 1

        if models is None:
            continue

        row = chol.iloc[p_loc]
        L = np.zeros((n_assets, n_assets))

        for a, b in lower_pairs:

            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
            X_now = row[lag_cols].values.reshape(1, -1)

            scaler_x = scalers_x[(a, b)]
            scaler_y = scalers_y[(a, b)]
            model = models[(a, b)]

            Xs_now = scaler_x.transform(X_now)
            y_scaled = model.predict(Xs_now)

            y_pred = scaler_y.inverse_transform(
                y_scaled.reshape(-1, 1)
            )[0, 0]

            i = asset_to_idx[a]
            j = asset_to_idx[b]
            L[i, j] = y_pred

        Sigma = L @ L.T
        Sigma = make_spd(Sigma, eps=RIDGE_EPS)

        if p not in proxy.index:
            continue

        true_vec = proxy.loc[p, proxy_cols].values
        S_true = vec_to_matrix(true_vec, n_assets)
        S_true = make_spd(S_true, eps=RIDGE_EPS)

        frob = frobenius_distance(S_true, Sigma)
        qlike = qlike_loss(S_true, Sigma, eps=RIDGE_EPS)

        if np.isfinite(frob) and np.isfinite(qlike):
            frob_list.append(frob)
            qlike_list.append(qlike)
            n_forecasts += 1

    return {
        "window": window,
        "n_estimators": n_estimators,
        "max_depth": max_depth,
        "learning_rate": learning_rate,
        "subsample": subsample,
        "colsample_bytree": colsample_bytree,
        "min_child_weight": min_child_weight,
        "reg_lambda": reg_lambda,
        "avg_qlike": np.mean(qlike_list) if len(qlike_list) > 0 else np.nan,
        "avg_frobenius": np.mean(frob_list) if len(frob_list) > 0 else np.nan,
        "n_forecasts": n_forecasts,
        "n_refits": n_refits
    }

# ---------------------------------------------------------
# OPTUNA OBJECTIVE
# ---------------------------------------------------------
trial_results = []

def objective(trial):
    # rolling window lengths that actually make sense in your context
    window = trial.suggest_categorical(
        "window",
        [126, 189, 252, 315, 378, 441, 504]
    )

    # boosting complexity: moderate range
    n_estimators = trial.suggest_int("n_estimators", 100, 500, step=50)
    max_depth = trial.suggest_int("max_depth", 2, 5)

    # conservative learning-rate range
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.08, log=True)

    # regularization via row/feature sampling
    subsample = trial.suggest_float("subsample", 0.6, 1.0, step=0.1)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.6, 1.0, step=0.1)

    # node regularization
    min_child_weight = trial.suggest_int("min_child_weight", 1, 15)
    reg_lambda = trial.suggest_float("reg_lambda", 0.1, 10.0, log=True)

    res = evaluate_xgb_spec(
        window=window,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        min_child_weight=min_child_weight,
        reg_lambda=reg_lambda
    )

    trial.set_user_attr("avg_frobenius", res["avg_frobenius"])
    trial.set_user_attr("n_forecasts", res["n_forecasts"])
    trial.set_user_attr("n_refits", res["n_refits"])

    trial_results.append(res)

    if not np.isfinite(res["avg_qlike"]):
        return 1e12

    return res["avg_qlike"]

# ---------------------------------------------------------
# RUN OPTUNA
# ---------------------------------------------------------
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study = optuna.create_study(direction="minimize", sampler=sampler)

study.optimize(objective, n_trials=N_TRIALS, show_progress_bar=True)

# ---------------------------------------------------------
# COLLECT RESULTS
# ---------------------------------------------------------
trials_df = study.trials_dataframe().copy()

# add user attrs explicitly
trials_df["avg_frobenius"] = [t.user_attrs.get("avg_frobenius", np.nan) for t in study.trials]
trials_df["n_forecasts"] = [t.user_attrs.get("n_forecasts", np.nan) for t in study.trials]
trials_df["n_refits"] = [t.user_attrs.get("n_refits", np.nan) for t in study.trials]

# rename Optuna value column for clarity
if "value" in trials_df.columns:
    trials_df = trials_df.rename(columns={"value": "avg_qlike"})

# clean parameter column names
rename_map = {
    "params_window": "window",
    "params_n_estimators": "n_estimators",
    "params_max_depth": "max_depth",
    "params_learning_rate": "learning_rate",
    "params_subsample": "subsample",
    "params_colsample_bytree": "colsample_bytree",
    "params_min_child_weight": "min_child_weight",
    "params_reg_lambda": "reg_lambda",
}
trials_df = trials_df.rename(columns=rename_map)

# keep the most useful columns
keep_cols = [
    "number",
    "state",
    "avg_qlike",
    "avg_frobenius",
    "window",
    "n_estimators",
    "max_depth",
    "learning_rate",
    "subsample",
    "colsample_bytree",
    "min_child_weight",
    "reg_lambda",
    "n_forecasts",
    "n_refits",
    "datetime_start",
    "datetime_complete",
    "duration",
]
keep_cols = [c for c in keep_cols if c in trials_df.columns]
trials_df = trials_df[keep_cols]

# sort the same way as your brute-force results
trials_df = trials_df.sort_values(
    ["avg_qlike", "avg_frobenius", "window"],
    ascending=[True, True, True]
).reset_index(drop=True)

# ---------------------------------------------------------
# SAVE + PRINT
# ---------------------------------------------------------

best_params_df = pd.DataFrame([study.best_params])
best_params_df["best_avg_qlike"] = study.best_value
best_params_df["best_avg_frobenius"] = trials_df.iloc[0]["avg_frobenius"] if len(trials_df) > 0 else np.nan

print("\n=========================================================")
print("OPTUNA XGBOOST SEARCH COMPLETE")
print("Forecast horizon:", TARGET_H)
print(f"Validation period: {VALIDATION_START} -> {VALIDATION_END}")
print(f"Refit every: {REFIT_EVERY} forecast origins")
print("=========================================================\n")

print("Top trials ranked by QLIKE:\n")
print(trials_df.head(20).to_string(index=False))

print("\n=========================================================")
print("BEST OPTUNA MODEL")
print("=========================================================")
print("Best avg QLIKE:", study.best_value)
print("Best params:")
for k, v in study.best_params.items():
    print(f"  {k}: {v}")


[I 2026-03-18 14:44:49,693] A new study created in memory with name: no-name-91b2435a-a8d2-459a-a412-8000f7822978


Loaded Cholesky data: ../proxy/realized_chol_target_h21_lag_5_10_21_63.csv
Loaded proxy data: ../proxy/realized_cov_h21.csv
Validation: 2019-01-02 00:00:00 -> 2020-12-02 00:00:00
Refit every: 21
Number of Optuna trials: 40



  0%|          | 0/40 [00:00<?, ?it/s]

[I 2026-03-18 14:45:40,444] Trial 0 finished with value: 125.35469362690019 and parameters: {'window': 189, 'n_estimators': 450, 'max_depth': 4, 'learning_rate': 0.04359666365651395, 'subsample': 0.6, 'colsample_bytree': 1.0, 'min_child_weight': 13, 'reg_lambda': 0.26587543983272705}. Best is trial 0 with value: 125.35469362690019.
[I 2026-03-18 14:45:55,921] Trial 1 finished with value: 42.73707310577088 and parameters: {'window': 504, 'n_estimators': 150, 'max_depth': 3, 'learning_rate': 0.021421886418348784, 'subsample': 0.8, 'colsample_bytree': 0.9, 'min_child_weight': 3, 'reg_lambda': 1.0677482709481354}. Best is trial 1 with value: 42.73707310577088.
[I 2026-03-18 14:46:33,656] Trial 2 finished with value: 48.41566661080341 and parameters: {'window': 504, 'n_estimators': 450, 'max_depth': 3, 'learning_rate': 0.01225199210177624, 'subsample': 0.9, 'colsample_bytree': 0.8, 'min_child_weight': 2, 'reg_lambda': 0.9780337016659408}. Best is trial 1 with value: 42.73707310577088.
[I 20

Forecasting scheme

In [13]:
import numpy as np
import pandas as pd
from xgboost import XGBRegressor
from sklearn.preprocessing import StandardScaler
from tqdm.auto import tqdm

# =========================================================
# FINAL XGBOOST-CHOLESKY COVARIANCE FORECAST
# Insert tuned parameters below
# Trains on rolling window and forecasts over test period
# Saves full covariance matrix forecast at each forecast origin
# =========================================================

# ---------------------------------------------------------
# SETTINGS: INSERT YOUR TUNED VALUES HERE
# ---------------------------------------------------------
TARGET_H = 21

CHOL_PATH = f"../proxy/realized_chol_target_h{TARGET_H}_lag_5_10_21_63.csv"
PROXY_PATH = f"../proxy/realized_cov_h{TARGET_H}.csv"
SAVE_FORECAST_PATH = f"../results/xgb_cov_forecast_h{TARGET_H}.csv"

ASSET_ORDER = [
    "US_EQUITY", "INTL_EQUITY", "US_BONDS", "INTL_BONDS",
    "US_REIT", "INTL_REIT", "GOLD", "BTC"
]

TEST_START = "2021-01-01"
TEST_END = None   # None = until last available date

# Insert tuned window here
TRAINING_WINDOW = 315

# Use the same lag structure as in tuning
LAGS = (10, 21, 63)

# Insert tuned refit frequency here
REFIT_EVERY = 1

RIDGE_EPS = 1e-8
RANDOM_STATE = 42
N_JOBS = -1

# ---------------------------------------------------------
# INSERT TUNED XGBOOST PARAMETERS HERE
# ---------------------------------------------------------
XGB_PARAMS = {
    "n_estimators": 100,
    "max_depth": 2,
    "learning_rate": 0.01,
    "subsample": 1,
    "colsample_bytree": 0.7,
    "min_child_weight": 15,
    "reg_lambda": 0.123,
    "objective": "reg:squarederror",
    "random_state": RANDOM_STATE,
    "n_jobs": N_JOBS
}

# ---------------------------------------------------------
# LOAD DATA
# ---------------------------------------------------------
chol = (
    pd.read_csv(CHOL_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .reset_index(drop=True)
)

proxy = (
    pd.read_csv(PROXY_PATH, parse_dates=["Date"])
    .sort_values("Date")
    .set_index("Date")
)

if TEST_END is None:
    TEST_END = chol["Date"].max()

test_dates = chol.loc[
    (chol["Date"] >= pd.Timestamp(TEST_START)) &
    (chol["Date"] <= pd.Timestamp(TEST_END)),
    "Date"
]

print("Loaded Cholesky data:", CHOL_PATH)
print("Loaded proxy data:", PROXY_PATH)
print("Test period:", test_dates.min(), "->", test_dates.max())
print("Training window:", TRAINING_WINDOW)
print("Refit every:", REFIT_EVERY)
print("XGB params:", XGB_PARAMS)
print()

# ---------------------------------------------------------
# HELPERS
# ---------------------------------------------------------
def build_lower_tri_pairs(asset_order):
    pairs = []
    for i, a in enumerate(asset_order):
        for j, b in enumerate(asset_order):
            if i >= j:
                pairs.append((a, b))
    return pairs

def build_full_pairs(asset_order):
    return [(a, b) for a in asset_order for b in asset_order]

def vec_to_matrix(vec, n_assets):
    M = np.asarray(vec, dtype=float).reshape(n_assets, n_assets)
    M = 0.5 * (M + M.T)
    return M

def frobenius_distance(S_true, H_pred):
    return float(np.linalg.norm(S_true - H_pred, ord="fro"))

def make_spd(M, eps=RIDGE_EPS):
    M = np.asarray(M, dtype=float)
    M = 0.5 * (M + M.T)
    eigvals = np.linalg.eigvalsh(M)
    min_eig = eigvals.min()
    if min_eig <= eps:
        M = M + np.eye(M.shape[0]) * (eps - min_eig + eps)
    return M

def qlike_loss(S_true, H_pred, eps=RIDGE_EPS):
    """
    Matrix QLIKE:
        tr(H_pred^{-1} S_true) - logdet(S_true) + logdet(H_pred) - n
    """
    S_true = make_spd(S_true, eps=eps)
    H_pred = make_spd(H_pred, eps=eps)

    n = S_true.shape[0]

    sign_s, logdet_s = np.linalg.slogdet(S_true)
    sign_h, logdet_h = np.linalg.slogdet(H_pred)

    if sign_s <= 0 or sign_h <= 0:
        return np.nan

    try:
        trace_term = np.trace(np.linalg.solve(H_pred, S_true))
    except np.linalg.LinAlgError:
        return np.nan

    return float(trace_term - logdet_s + logdet_h - n)

# ---------------------------------------------------------
# STRUCTURE
# ---------------------------------------------------------
lower_pairs = build_lower_tri_pairs(ASSET_ORDER)
full_pairs = build_full_pairs(ASSET_ORDER)

n_assets = len(ASSET_ORDER)
asset_to_idx = {a: i for i, a in enumerate(ASSET_ORDER)}

forecast_cols = [f"cov_{a}__{b}" for a, b in full_pairs]
proxy_cols = forecast_cols.copy()

missing_proxy_cols = [c for c in proxy_cols if c not in proxy.columns]
if missing_proxy_cols:
    raise ValueError(f"Missing proxy columns, e.g. {missing_proxy_cols[:5]}")

# ---------------------------------------------------------
# FORECAST LOOP
# ---------------------------------------------------------
models = None
scalers_x = None
scalers_y = None
last_refit_loc = None

forecast_rows = []
eval_rows = []

n_refits = 0
n_forecasts = 0

for p in tqdm(test_dates, desc=f"XGBoost forecast (h={TARGET_H})"):
    p_loc = chol.index[chol["Date"] == p][0]

    should_refit = (
        models is None
        or last_refit_loc is None
        or (p_loc - last_refit_loc) >= REFIT_EVERY
    )

    if should_refit:
        train_end = p_loc - TARGET_H
        train_start = train_end - TRAINING_WINDOW + 1

        if train_start < 0:
            continue

        train = chol.iloc[train_start:train_end + 1]

        models = {}
        scalers_x = {}
        scalers_y = {}

        for a, b in lower_pairs:
            target_col = f"target_chol_{a}__{b}"
            lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]

            missing_cols = [c for c in lag_cols + [target_col] if c not in train.columns]
            if missing_cols:
                raise ValueError(f"Missing columns for {(a, b)}: {missing_cols}")

            X = train[lag_cols].values
            y = train[target_col].values.reshape(-1, 1)

            scaler_x = StandardScaler()
            scaler_y = StandardScaler()

            Xs = scaler_x.fit_transform(X)
            ys = scaler_y.fit_transform(y).ravel()

            model = XGBRegressor(**XGB_PARAMS)
            model.fit(Xs, ys)

            models[(a, b)] = model
            scalers_x[(a, b)] = scaler_x
            scalers_y[(a, b)] = scaler_y

        last_refit_loc = p_loc
        n_refits += 1

    if models is None:
        continue

    row = chol.iloc[p_loc]
    L = np.zeros((n_assets, n_assets))

    for a, b in lower_pairs:
        lag_cols = [f"lag{lag}_chol_{a}__{b}" for lag in LAGS]
        X_now = row[lag_cols].values.reshape(1, -1)

        scaler_x = scalers_x[(a, b)]
        scaler_y = scalers_y[(a, b)]
        model = models[(a, b)]

        Xs_now = scaler_x.transform(X_now)
        y_scaled = model.predict(Xs_now)

        y_pred = scaler_y.inverse_transform(y_scaled.reshape(-1, 1))[0, 0]

        i = asset_to_idx[a]
        j = asset_to_idx[b]
        L[i, j] = y_pred

    Sigma = L @ L.T
    Sigma = make_spd(Sigma, eps=RIDGE_EPS)

    # save forecast row
    out_row = {"Date": p}
    for i, a in enumerate(ASSET_ORDER):
        for j, b in enumerate(ASSET_ORDER):
            out_row[f"cov_{a}__{b}"] = Sigma[i, j]
    forecast_rows.append(out_row)

    # evaluation against proxy if available
    if p in proxy.index:
        true_vec = proxy.loc[p, proxy_cols].values
        S_true = vec_to_matrix(true_vec, n_assets)
        S_true = make_spd(S_true, eps=RIDGE_EPS)

        frob = frobenius_distance(S_true, Sigma)
        qlike = qlike_loss(S_true, Sigma, eps=RIDGE_EPS)

        eval_rows.append({
            "Date": p,
            "frobenius": frob,
            "qlike": qlike
        })

    n_forecasts += 1

# ---------------------------------------------------------
# BUILD OUTPUTS
# ---------------------------------------------------------
forecast_df = pd.DataFrame(forecast_rows).sort_values("Date").reset_index(drop=True)
eval_df = pd.DataFrame(eval_rows).sort_values("Date").reset_index(drop=True)

# ---------------------------------------------------------
# SAVE
# ---------------------------------------------------------
forecast_df.to_csv(SAVE_FORECAST_PATH, index=False)



# ---------------------------------------------------------
# SUMMARY
# ---------------------------------------------------------
print("\n=========================================================")
print("DONE")
print("=========================================================")
print("Saved forecasts to:", SAVE_FORECAST_PATH)
print("Forecast rows:", len(forecast_df))
print("Refits:", n_refits)

if len(eval_df) > 0:
    print("\nAverage evaluation over available proxy dates:")
    print("Avg QLIKE     :", eval_df["qlike"].mean())
    print("Avg Frobenius :", eval_df["frobenius"].mean())
else:
    print("\nNo evaluation rows were computed.")

print("\nForecast head:")
print(forecast_df.head())

if len(eval_df) > 0:
    print("\nEvaluation head:")
    print(eval_df.head())

Loaded Cholesky data: ../proxy/realized_chol_target_h21_lag_5_10_21_63.csv
Loaded proxy data: ../proxy/realized_cov_h21.csv
Test period: 2021-01-04 00:00:00 -> 2025-12-02 00:00:00
Training window: 315
Refit every: 1
XGB params: {'n_estimators': 100, 'max_depth': 2, 'learning_rate': 0.01, 'subsample': 1, 'colsample_bytree': 0.7, 'min_child_weight': 15, 'reg_lambda': 0.123, 'objective': 'reg:squarederror', 'random_state': 42, 'n_jobs': -1}



XGBoost forecast (h=21):   0%|          | 0/1235 [00:00<?, ?it/s]


DONE
Saved forecasts to: ../results/xgb_cov_forecast_h21.csv
Forecast rows: 1235
Refits: 1235

Average evaluation over available proxy dates:
Avg QLIKE     : 5.3654267561325435
Avg Frobenius : 0.000959627852965636

Forecast head:
        Date  cov_US_EQUITY__US_EQUITY  cov_US_EQUITY__INTL_EQUITY  \
0 2021-01-04                  0.000090                    0.000079   
1 2021-01-05                  0.000100                    0.000085   
2 2021-01-06                  0.000167                    0.000108   
3 2021-01-07                  0.000167                    0.000108   
4 2021-01-08                  0.000167                    0.000108   

   cov_US_EQUITY__US_BONDS  cov_US_EQUITY__INTL_BONDS  cov_US_EQUITY__US_REIT  \
0             6.195659e-07                  -0.000003                0.000117   
1             6.790217e-07                  -0.000003                0.000124   
2             8.532865e-07                  -0.000003                0.000161   
3             8.653070e-